# Which cell types changes the most across disease (per dataset)

https://pertpy.readthedocs.io/en/stable/tutorials/notebooks/augur.html

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

#### Libraries

In [ ]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad         # Core data structure for single-cell data
import scanpy as sc          # Analysis and visualization of single-cell data
import pertpy as pt          # Augur analysis
import gc

from itertools import combinations   # Bulding iterations

# parallel processing
from joblib import parallel_backend  # Parallel computing support

import pandas as pd

# Miscellaneous utilities
import warnings              # Suppress or manage warnings
warnings.filterwarnings("ignore")

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as ma                    # Custom AnnData utilities

#### Parameters

In [ ]:
# Directories
ma.create_directories(dir_path = str(here('data/cell_type_prioritization')))
ma.create_directories(dir_path = str(here('data/cell_type_prioritization/files')))
ma.create_directories(dir_path = str(here('data/cell_type_prioritization/plot')))

#### Custom paths

In [ ]:
# Paths
base_dir = str(here('data/cell_type_prioritization/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 

anndata_dir = str(here('data/anndata/'))

#### Load

In [ ]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"), backed = 'r')

#### Setup

In [ ]:
anno_key   = "cell_type"
sample_key = "ic_id_platform_adjusted_sample"
donor_key  = "ic_id_donor_overall"
dataset_key  = "ic_id_dataset"
disease_key = "disease_hba1c"

#### Remove multiple donors

In [ ]:
# Find donors appearing in multiple datasets
donor_dataset_counts = adata.obs.groupby(donor_key)[dataset_key].nunique()
multi_dataset_donors = donor_dataset_counts[donor_dataset_counts > 1].index

print(f"Found {len(multi_dataset_donors)} donors in multiple datasets")

# Check which multi-dataset donors have 10x available
is_multi = adata.obs[donor_key].isin(multi_dataset_donors)
is_10x = adata.obs['library_prep'].str.contains('10x', case=False)

multi_with_10x = adata.obs[is_multi & is_10x][donor_key].nunique()
multi_without_10x = len(multi_dataset_donors) - multi_with_10x

print(f"  - {multi_with_10x} donors have 10x available (will prioritize)")
print(f"  - {multi_without_10x} donors only in smart-seq (will not keep those)")

# Keep: single-dataset donors + 10x versions of multi-dataset donors
keep = ~is_multi | is_10x
adata = adata[keep].copy(filename='AH_combined.h5ad')

print(f"Removed {(~keep).sum()} observations")

# Verify filtering worked
remaining_multi = adata.obs.groupby(donor_key)[dataset_key].nunique()
n_remaining_multi = (remaining_multi > 1).sum()
assert n_remaining_multi == 0, f"Error: {n_remaining_multi} donors still appear in multiple datasets!"
print("Filtering successful: all donors now in single dataset")
print("Number of donors: ", len(remaining_multi))

del is_multi, is_10x, multi_with_10x, multi_without_10x, keep, remaining_multi, n_remaining_multi
gc.collect()

#### Cell type prioritization

In [ ]:
datasets = adata.obs[dataset_key].unique().tolist()
ag_rfc = pt.tl.Augur("random_forest_classifier")

augur_summary = []
augur_full_results = []

for dset in datasets:
    mask = adata.obs[dataset_key] == dset
    adata_sub = adata[mask].to_memory()
    cats = adata_sub.obs[disease_key].cat.categories.tolist()

    if len(cats) < 2:
        print(f"  Skipping {dset} - only one disease category")
        continue

    from itertools import combinations
    contrasts = list(combinations(cats, 2))

    for cond_a, cond_b in contrasts:
        contrast_name = f"{cond_a}_vs_{cond_b}"
        try:
            loaded_data = ag_rfc.load(
                input=adata_sub,
                cell_type_col=anno_key,
                label_col=disease_key,
                condition_label=cond_a,
                treatment_label=cond_b,
                layer="counts",
            )
            v_adata, v_results = ag_rfc.predict(loaded_data, min_cells=20, random_state=42)

            # summary metrics (mean per cell type)
            results = v_results["summary_metrics"].copy()
            results["dataset"] = dset
            results["contrast"] = contrast_name
            augur_summary.append(results)

            # full CV results (per fold/subsample, per cell type) — used for TE/seTE in R
            full_res = v_results["full_results"].copy()
            full_res["dataset"] = dset
            full_res["contrast"] = contrast_name
            augur_full_results.append(full_res)

            del v_adata, v_results
        except Exception as e:
            print(f"  Skipping {dset}__{contrast_name}: {e}")

    del adata_sub

# combined outputs, tagged with dataset + contrast for downstream splitting
if augur_summary:
    pd.concat(augur_summary, ignore_index=False).to_csv(
        os.path.join(files_dir, "augur_summary_metrics.csv"), index=True, index_label="metric"
    )

if augur_full_results:
    pd.concat(augur_full_results, ignore_index=True).to_csv(
        os.path.join(files_dir, "augur_full_results.csv"), index=False
    )